In [1]:
from glob import glob
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import ttest_ind


/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
csv_files  = glob(os.path.join("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/crop_based/seg_results/","*.csv"))
pd.read_csv(csv_files[0]).head(2)

,Unnamed: 0,slide_id,img_name,area,perimeter,eccentricity,major_axis_length,minor_axis_length,solidity,extent,aspect_ratio,circularity,bbox_xmin,bbox_xmax,bbox_ymin,bbox_ymax,centroid_x,centroid_y,label,class
0,0,PD158_C110-115_EntCx David Menassa.svs,PD158_C110-115_EntCx David Menassa.svs/69627x_...,201,56.870058,0.780396,21.009171,13.136720,0.881579,0.628125,1.599271,0.780978,719,739,570,586,729.442786,578.711443,1,2
1,1,PD158_C110-115_EntCx David Menassa.svs,PD158_C110-115_EntCx David Menassa.svs/69627x_...,34,21.278175,0.890169,10.085081,4.595071,0.871795,0.618182,2.194760,0.943670,992,1003,762,767,997.735294,763.558824,2,2


In [4]:
df_list =  []
for i in range(len(csv_files)):
    grp_by_class= pd.read_csv(csv_files[i]).groupby(["slide_id","class"]).agg({"area":["mean","std"],"perimeter":["mean","std"],"eccentricity":["mean","std"],
                                                                "major_axis_length":["mean","std"],"minor_axis_length":["mean","std"],
                                                                "solidity":["mean","std"],"extent":["mean","std"],"aspect_ratio":["mean","std"],
                                                                "circularity":["mean","std"], "img_name":["count"]})
    grp_by_class.columns = ["area_mean",'area_std','perimeter_mean','perimeter_std','eccentricity_mean',
                  'eccentricity_std',  'major_axis_length_mean', 'major_axis_length_std', 'minor_axis_length_mean',
                  'minor_axis_length_std','solidity_mean', 'solidity_std', 'extent_mean',  'extent_std', 'aspect_ratio_mean',
                  'aspect_ratio_std', 'circularity_mean', 'circularity_std', "lb_count"]
    
    grp_by_class = grp_by_class.reset_index()
    grp_by_slide = pd.pivot(grp_by_class, columns =  ["class"], index="slide_id")
    col_names = []
    mapping = {1:"mature_lb",2:"pre_lb",3:"neg_lb"}
    for column in grp_by_slide.columns:
        col_names.append(column[0]+"_"+mapping[column[1]])
    grp_by_slide.columns = col_names
    grp_by_slide= grp_by_slide.reset_index()
    df_list.append(grp_by_slide)

In [5]:
all_df_features  =  pd.concat(df_list)

In [6]:
oxford_data = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/prov-gigapath/data/antibodies_data/data_imperial2.csv")

In [7]:
oxford_data_feat =  pd.merge(oxford_data,all_df_features, on="slide_id", how="left")

In [8]:
oxford_data_feat.head(2)

,slide_id,brain_region,pat_id,anitibody,LBD_flag,label,Brain_bank,area_mean_mature_lb,area_mean_pre_lb,area_mean_neg_lb,...,aspect_ratio_std_neg_lb,circularity_mean_mature_lb,circularity_mean_pre_lb,circularity_mean_neg_lb,circularity_std_mature_lb,circularity_std_pre_lb,circularity_std_neg_lb,lb_count_mature_lb,lb_count_pre_lb,lb_count_neg_lb
0,PD134-DLB--Amygdala-C110-115.svs,Amygdala,PD134,C110-115,DLB,1,Oxford,547.247741,235.936411,359.281426,...,1.058582,1.008514,1.152042,1.016600,2.588069,2.857844,2.082594,18261.0,17393.0,18154.0
1,PD134-DLB--EntCx-C110-115.svs,EntCx,PD134,C110-115,DLB,1,Oxford,476.007864,220.410873,432.484103,...,1.076915,1.054892,1.157449,1.033833,2.769480,2.948588,2.273730,10427.0,14384.0,14846.0


In [9]:
oxford_data_feat.columns

Index(['slide_id', 'brain_region', 'pat_id', 'anitibody', 'LBD_flag', 'label',
       'Brain_bank', 'area_mean_mature_lb', 'area_mean_pre_lb',
       'area_mean_neg_lb', 'area_std_mature_lb', 'area_std_pre_lb',
       'area_std_neg_lb', 'perimeter_mean_mature_lb', 'perimeter_mean_pre_lb',
       'perimeter_mean_neg_lb', 'perimeter_std_mature_lb',
       'perimeter_std_pre_lb', 'perimeter_std_neg_lb',
       'eccentricity_mean_mature_lb', 'eccentricity_mean_pre_lb',
       'eccentricity_mean_neg_lb', 'eccentricity_std_mature_lb',
       'eccentricity_std_pre_lb', 'eccentricity_std_neg_lb',
       'major_axis_length_mean_mature_lb', 'major_axis_length_mean_pre_lb',
       'major_axis_length_mean_neg_lb', 'major_axis_length_std_mature_lb',
       'major_axis_length_std_pre_lb', 'major_axis_length_std_neg_lb',
       'minor_axis_length_mean_mature_lb', 'minor_axis_length_mean_pre_lb',
       'minor_axis_length_mean_neg_lb', 'minor_axis_length_std_mature_lb',
       'minor_axis_length_std_p

In [10]:
oxford_data_feat["LBD_flag"].value_counts()

LBD_flag
DLB    34
PDD    34
Name: count, dtype: int64

In [11]:
oxford_data_feat.groupby(["LBD_flag"])["lb_count_mature_lb"].mean()

LBD_flag
DLB    17767.382353
PDD     9540.424242
Name: lb_count_mature_lb, dtype: float64

In [12]:
oxford_data_feat.groupby(["LBD_flag"])["lb_count_pre_lb"].mean()

LBD_flag
DLB    13847.500000
PDD    19744.969697
Name: lb_count_pre_lb, dtype: float64

In [13]:
oxford_data_feat.groupby(["LBD_flag"])["lb_count_neg_lb"].mean()

LBD_flag
DLB    12555.529412
PDD    20181.969697
Name: lb_count_neg_lb, dtype: float64

In [14]:
oxford_data_feat.groupby(["LBD_flag"])["area_mean_mature_lb"].mean()

LBD_flag
DLB    482.588585
PDD    510.393066
Name: area_mean_mature_lb, dtype: float64

In [15]:
oxford_data_feat.groupby(["LBD_flag"])["circularity_mean_mature_lb"].mean()

LBD_flag
DLB    1.048320
PDD    1.041182
Name: circularity_mean_mature_lb, dtype: float64

In [16]:
oxford_data_feat.groupby(["LBD_flag"])["eccentricity_mean_mature_lb"].mean()

LBD_flag
DLB    0.735752
PDD    0.753425
Name: eccentricity_mean_mature_lb, dtype: float64

In [17]:
oxford_data_feat.groupby(["LBD_flag",'brain_region'])["lb_count_mature_lb"].mean()

LBD_flag  brain_region
DLB       Amygdala        26071.750000
          EntCx           13140.214286
          Striatum        13408.375000
PDD       Amygdala        11826.705882
          EntCx            6328.461538
          Striatum        10503.333333
Name: lb_count_mature_lb, dtype: float64

In [18]:
oxford_data_feat.groupby(["LBD_flag",'anitibody'])["lb_count_mature_lb"].mean()

LBD_flag  anitibody
DLB       C110-115     18728.214286
          C34-45       17094.800000
PDD       C110-115      6980.391304
          C34-45       15428.500000
Name: lb_count_mature_lb, dtype: float64

In [20]:
oxford_data_feat.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/crop_based/analysis/slide_features.csv")

In [37]:
columns_to_vis = oxford_data_feat.columns[7:].values

In [47]:
imp= []
for col in columns_to_vis:
    print(oxford_data_feat.groupby(["LBD_flag"])[col].mean())
    res= ttest_ind(oxford_data_feat[oxford_data_feat["LBD_flag"]=="PDD"][col].values, oxford_data_feat[oxford_data_feat["LBD_flag"]=="DLB"][col].values,nan_policy='omit')
    print(res)
    if res.pvalue<0.05:
        imp.append(col)
        

LBD_flag
DLB    482.588585
PDD    510.393066
Name: area_mean_mature_lb, dtype: float64
Ttest_indResult(statistic=0.6034107120436962, pvalue=0.5483354352003176)
LBD_flag
DLB    189.351592
PDD    188.737810
Name: area_mean_pre_lb, dtype: float64
Ttest_indResult(statistic=-0.08413852901090131, pvalue=0.9332048936523412)
LBD_flag
DLB    3691.354405
PDD    7714.417578
Name: area_mean_neg_lb, dtype: float64
Ttest_indResult(statistic=1.671555259501149, pvalue=0.0994200557915516)
LBD_flag
DLB    601.664016
PDD    724.082278
Name: area_std_mature_lb, dtype: float64
Ttest_indResult(statistic=1.2989389711438102, pvalue=0.1985538721158314)
LBD_flag
DLB    212.015844
PDD    192.831587
Name: area_std_pre_lb, dtype: float64
Ttest_indResult(statistic=-3.514815358379338, pvalue=0.0008070882176456953)
LBD_flag
DLB    17619.895592
PDD    39815.098439
Name: area_std_neg_lb, dtype: float64
Ttest_indResult(statistic=2.09228948255126, pvalue=0.040322195469026094)
LBD_flag
DLB    79.695205
PDD    89.517914
Na

In [48]:
imp

['area_std_pre_lb',
 'area_std_neg_lb',
 'perimeter_std_pre_lb',
 'eccentricity_mean_mature_lb',
 'eccentricity_mean_pre_lb',
 'eccentricity_std_pre_lb',
 'minor_axis_length_mean_neg_lb',
 'minor_axis_length_std_pre_lb',
 'minor_axis_length_std_neg_lb',
 'solidity_mean_mature_lb',
 'solidity_mean_neg_lb',
 'solidity_std_pre_lb',
 'solidity_std_neg_lb',
 'extent_mean_pre_lb',
 'extent_std_mature_lb',
 'extent_std_pre_lb',
 'aspect_ratio_mean_mature_lb',
 'aspect_ratio_mean_pre_lb',
 'aspect_ratio_std_mature_lb',
 'circularity_mean_pre_lb',
 'circularity_mean_neg_lb',
 'circularity_std_pre_lb',
 'lb_count_mature_lb',
 'lb_count_pre_lb']